# Power grid resilience

Part A ranks nodes by a heuristic vulnerability score. Part B trains a tabular Q-learning agent to redistribute load from overloaded nodes, then compares it with four fixed-rule baselines.

Design: [`docs/design.md`](../docs/design.md)

In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd
import matplotlib.pyplot as plt

import viz
from grid import PowerGrid
from cascade import CascadeSimulator
from heuristic import VulnerabilityAnalyzer, plot_tiers, plot_validation
from env import GridEnv
from agent import QLearningAgent, Trainer
from evaluate import (Evaluator, QPolicy, DoNothingPolicy, EvenSplitPolicy,
                      GreedySparePolicy, ShedPolicy)

viz.apply_style()
pd.set_option("display.precision", 2)

DATA = "../data/power_grid_dataset_with_cascade_failures.csv"
grid = PowerGrid.from_csv(DATA, k=4)
print(f"{len(grid.nodes)} nodes, {len(grid.edges())} links")

## Part A: Vulnerability detection

Scores are computed on the post-outage grid: the 32 damaged nodes are offline and their load has moved to their active neighbours.

In [ ]:
outage = CascadeSimulator(grid.copy())
outage.apply_initial_outage()
post = outage.grid

analyzer = VulnerabilityAnalyzer(w_stress=0.5, w_spill=0.35, w_expo=0.15)
ranked = analyzer.rank(post)
print(ranked["tier"].value_counts().to_string())
ranked.head(15)

In [ ]:
plot_tiers(post, ranked)
plt.show()

### Validation

Does the score predict what the uncontrolled cascade actually does? Each active node is labelled with the wave in which it fails (or *survived*). Critical nodes fail in wave 1 by definition, so the non-Critical figures are the more informative check.

In [ ]:
val = analyzer.validate(post, ranked)
print(f"Uncontrolled cascade: {val['cascade_failures']} of {val['cascade_failures'] + val['survivors']} active nodes fail in {val['waves']} waves")
print(f"All active nodes:   Spearman rho = {val['spearman']:.2f}, AUC = {val['auc']:.2f}")
print(f"Non-Critical nodes: Spearman rho = {val['spearman_non_critical']:.2f}, AUC = {val['auc_non_critical']:.2f}")
plot_validation(val)
plt.show()

## Part B: Q-learning load redistribution

### Training

5,000 episodes; ε decays linearly from 1.0 to 0.05 over the first 4,000.

In [ ]:
env = GridEnv(grid, max_rounds=20, demand_noise=0.15, extra_fail_prob=0.5)
agent = QLearningAgent(env.encoder.n_states, alpha=0.1, gamma=0.9,
                       eps_start=1.0, eps_end=0.05, eps_decay_episodes=4000, seed=0)
history = Trainer(env, agent, episodes=5000, seed_offset=0).train(log_every=500)
print(f"States visited: {(agent.visits.sum(axis=1) > 0).sum()} / {env.encoder.n_states}")

In [ ]:
Evaluator.plot_training(history)
plt.show()

In [ ]:
Evaluator.plot_policy(agent, env.encoder)
plt.show()

### Evaluation

All policies run on the same 500 test scenarios (seeds 100,000–100,499, never seen in training). The agent acts greedily.

In [ ]:
policies = [QPolicy(agent), DoNothingPolicy(), EvenSplitPolicy(), GreedySparePolicy(), ShedPolicy()]
evaluator = Evaluator(env)
results = evaluator.run(policies)
Evaluator.summary(results)

In [ ]:
Evaluator.plot_results(results)
plt.show()

### One test episode, step by step

In [ ]:
seed = 100_000
evaluator.plot_episode(QPolicy(agent), seed)
plt.show()
evaluator.plot_episode(DoNothingPolicy(), seed)
plt.show()